# Resultados-Únicos-Saber-11

El conjunto de datos Resultados Únicos Saber 11 contiene información detallada de los estudiantes que presentaron las pruebas Saber 11 en Colombia, junto con variables asociadas a su contexto educativo, geográfico y socioeconómico.

Este conjunto de datos es relevante para el desarrollo del proyecto, dado que uno de los principales indicadores de interés del problema de negocio corresponde a los resultados de las pruebas Saber 11. En particular, permite analizar el desempeño académico de los estudiantes y relacionarlo con factores como el acceso a internet, las condiciones socioeconómicas y las características del entorno educativo, especialmente en el departamento de Boyacá, que constituye el foco del análisis.

Fuente: https://www.datos.gov.co/Educaci-n/Resultados-nicos-Saber-11/kgxf-xxbe/about_data 

## 1. Data Understanding

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, upper, lower, when, regexp_replace

### 1.1 Carga Inicial de Datos

In [0]:


dfPruebasSaber = spark.read.csv("/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/Resultados-Únicos-Saber-11.csv",header=True, inferSchema=True)
dfPruebasSaber.createOrReplaceTempView("dfPruebasSaber")
display(dfPruebasSaber.limit(10))

### 1.2 Descripción de Datos

In [0]:
dfPruebasSaber.printSchema()
print(dfPruebasSaber.columns)

In [0]:
print("Número de filas:", dfPruebasSaber.count())
print("Número de columnas:", len(dfPruebasSaber.columns))

El dataset contiene 220,942 registros y 51 variables.

**Descripción general del conjunto de datos**

Este dataset incluye tanto características del estudiante (por ejemplo, género, lugar de residencia y condiciones familiares), como información del establecimiento educativo (ubicación, jornada, naturaleza y características institucionales), así como los resultados obtenidos en las distintas áreas evaluadas, tales como matemáticas, lectura crítica, ciencias naturales, sociales e inglés.

Adicionalmente, se dispone de un puntaje global que resume el desempeño general del estudiante en la prueba, lo cual permite realizar comparaciones y análisis agregados entre diferentes grupos poblacionales. Las variables presentes en el conjunto de datos se pueden agrupar en cuatro grandes categorías: variables del estudiante (prefijo `ESTU_`), variables del colegio (prefijo `COLE_`), variables del contexto familiar (prefijo `FAMI_`), y variables de resultados académicos (prefijo `PUNT_` y `DESEMP_`).

## 2. Exploración de Datos

### 2.1 Filtrar dataset - registros en Boyacá

Todos los registros relacionados con Boyacá, ya sea porque el estudiante estudia allí o porque vive allí.

In [0]:
# Filtrar Boyacá
dfBoyaca = dfPruebasSaber.filter(
  (col("COLE_DEPTO_UBICACION") == "BOYACA") | (col("ESTU_DEPTO_RESIDE") == "BOYACA")
)
print("Registros en Boyacá:", dfBoyaca.count())
display(dfBoyaca.limit(10))

Se incluyen tanto los estudiantes que presentan la prueba en instituciones ubicadas en Boyacá, como aquellos que residen en este departamento, aunque estudien en otra región. 
Este enfoque permite capturar de forma más amplia la población relacionada con el departamento, teniendo en cuenta posibles dinámicas de movilidad estudiantil entre regiones. Como resultado del filtrado, se obtuvieron 212,840 registros asociados a Boyacá.

### 2.2 Estadísticas descriptivas generales

In [0]:
#Estadísticas de variables numéricas
display(dfBoyaca.describe())

El puntaje global tiene un promedio cercano a 264.9 y una desviación estándar de aproximadamente 46.2, lo que indica una variabilidad moderada en el desempeño de los estudiantes. También, se identifican valores mínimos y máximos que reflejan tanto casos de bajo rendimiento como de alto desempeño, siendo el puntaje global máximo de 471. Sin embargo, se logra evidenciar valores atípicos o inconsistentes en algunas variables, como puntajes negativos en inglés (-1.0), lo cual sugiere la presencia de registros inválidos o codificaciones especiales que deberán ser tratados en etapas posteriores de limpieza.

### 2.3 Promedio de puntaje global por periodo

In [0]:
df01 = dfBoyaca.groupBy("PERIODO").agg(
    F.avg("PUNT_GLOBAL").alias("promedio_puntaje_global"),
    F.count("*").alias("cantidad_estudiantes")
).orderBy("PERIODO")

display(df01)

El análisis del puntaje global promedio por periodo permite identificar cambios en la disponibilidad y comportamiento de los datos a lo largo del tiempo. Se observa que para varios periodos anteriores al año 2014, el valor del puntaje global no se encuentra registrado (valores nulos), lo que indica que esta variable no estaba disponible o no fue incluida en versiones anteriores del dataset.

Adicionalmente, se evidencia una alta variabilidad en la cantidad de estudiantes registrados por periodo, con valores que van desde menos de 100 registros hasta más de 33,000 en algunos casos. Esta diferencia puede estar asociada a cambios en la cobertura de la prueba, en la disponibilidad de los datos o en los procesos de consolidación de la información a lo largo de los años, como consecuencia, los periodos con pocos registros pueden no ser representativos para análisis estadísticos robustos.

En cuanto al comportamiento del puntaje global, se observa que los promedios se mantienen en un rango relativamente estable entre 240 y 270 puntos en los periodos más recientes, lo que sugiere una cierta estabilidad en el desempeño promedio de los estudiantes en Boyacá.



### 2.4 Puntaje promedio por municipio

In [0]:
df02 = dfBoyaca.groupBy("COLE_MCPIO_UBICACION").agg(
    F.avg("PUNT_GLOBAL").alias("promedio_puntaje"),
    F.count("*").alias("cantidad")
).orderBy(F.col("promedio_puntaje").desc())

display(df02.limit(10))

El análisis del puntaje global promedio por municipio, realizado sobre el conjunto filtrado para Boyacá, evidencia la presencia de municipios que no pertenecen a este departamento (por ejemplo, Medellín, Cali o Bogotá D.C.). Este comportamiento se explica por el criterio de filtrado aplicado, el cual incluye registros en los que el estudiante reside en Boyacá (`ESTU_DEPTO_RESIDE` = "BOYACA") aun cuando el establecimiento educativo se ubique en otro departamento.

En consecuencia, el conjunto de datos contiene tanto municipios correspondientes a la ubicación del colegio como a la residencia del estudiante, lo que introduce una mezcla de contextos geográficos. Adicionalmente, se identifican inconsistencias en la escritura de algunos municipios (por ejemplo, presencia o ausencia de tildes), lo cual sugiere la necesidad de procesos de estandarización en etapas posteriores de limpieza.

Para análisis más específicos por municipio dentro de Boyacá, será necesario modificar el filtrado considerando únicamente la ubicación del establecimiento educativo o la residencia del estudiante, dependiendo de como enfoquemos el estudio.


### 2.5 Puntaje promedio por naturaleza del colegio

In [0]:
df03 = dfBoyaca.groupBy("COLE_NATURALEZA").agg(
    F.avg("PUNT_GLOBAL").alias("promedio_puntaje"),
    F.count("*").alias("cantidad")
).orderBy(F.col("promedio_puntaje").desc())

display(df03)

Databricks visualization. Run in Databricks to view.

Los colegios no oficiales presentan un puntaje promedio superior (≈284.8) en comparación con los colegios oficiales (≈260.2). Asimismo, se observa que la gran mayoría de los registros corresponden a colegios oficiales, lo cual es consistente con la cobertura del sistema educativo público.

### 2.6 Puntaje por jornada

In [0]:
df04 = dfBoyaca.groupBy("COLE_JORNADA").agg(
    F.avg("PUNT_GLOBAL").alias("promedio_puntaje"),
    F.count("*").alias("cantidad")
)
display(df04)

Databricks visualization. Run in Databricks to view.

Los colegios con jornadas en la mañana, completa y única son los que presentan un puntaje promedio superior a los demás (≈267-268), seguidos de cerca por los colegios en jornada de la tarde (≈255).

### 2.7 Puntaje por zona (urbano vs rural)

In [0]:
df05 = dfBoyaca.groupBy("COLE_AREA_UBICACION").agg(
    F.avg("PUNT_GLOBAL").alias("promedio_puntaje"),
    F.count("*").alias("cantidad")
)
display(df05)

Databricks visualization. Run in Databricks to view.

Los colegios en zona urbana presentan un puntaje promedio superior (≈268) a los de zona rural (≈248). También se evidencia que la mayoría de los colegios se encuentran en zonas urbanas.

### 2.8 Relación con acceso a internet en el hogar

In [0]:
df06 = dfBoyaca.groupBy("FAMI_TIENEINTERNET").agg(
    F.avg("PUNT_GLOBAL").alias("promedio_puntaje"),
    F.count("*").alias("cantidad")
)
display(df06)

Databricks visualization. Run in Databricks to view.

En los hogares que tienen acceso a internet se presenta un puntaje promedio mayor (≈278) que en los hogares que no cuentan con acceso a internet (≈252), siendo estos últimos los que tienen mayor cantidad de registros.

### 2.9 Relación con computador en el hogar

In [0]:
df07 = dfBoyaca.groupBy("FAMI_TIENECOMPUTADOR").agg(
    F.avg("PUNT_GLOBAL").alias("promedio_puntaje"),
    F.count("*").alias("cantidad")
)
display(df07)

Databricks visualization. Run in Databricks to view.

También se presenta una diferencia en el puntaje de promedio entre los hogares que tienen computador (≈276) y los que no lo tienen (≈250).

### 2.10 Puntaje por estrato

In [0]:
df08 = dfBoyaca.groupBy("FAMI_ESTRATOVIVIENDA").agg(
    F.avg("PUNT_GLOBAL").alias("promedio_puntaje"),
    F.count("*").alias("cantidad")
).orderBy("FAMI_ESTRATOVIVIENDA")
display(df08)

Pareciera que, a medida que el estrato de la vivienda sube, también lo hace el puntaje promedio, logrando evidenciar una relación proporcional entre el nivel de estrato y el puntaje en las pruebas. Aunque debemos resaltar que en el nivel de Estrato 6, este no sigue la tendencia puesto que baja el puntaje promedio.

### 2.11 Promedio por materias (para entender estructura)

In [0]:
df09 = dfBoyaca.select(
    F.avg("PUNT_LECTURA_CRITICA").alias("lectura"),
    F.avg("PUNT_MATEMATICAS").alias("matematicas"),
    F.avg("PUNT_C_NATURALES").alias("ciencias"),
    F.avg("PUNT_SOCIALES_CIUDADANAS").alias("sociales"),
    F.avg("PUNT_INGLES").alias("ingles")
)
display(df09)

Los resultados se encuentran en rangos relativamente similares entre las distintas materias, con valores cercanos entre 51 y 54 puntos. Se observa un desempeño ligeramente superior en lectura crítica y ciencias naturales, mientras que el puntaje más bajo corresponde al área de inglés.

## 3. Reporte de Calidad de Datos

### 3.1 Conteo de valores nulos

In [0]:
import pandas as pd
total_registros = dfBoyaca.count()

dfNulos = dfBoyaca.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in dfBoyaca.columns
])
nulos_dict = dfNulos.collect()[0].asDict()

reporte_nulos = pd.DataFrame(list(nulos_dict.items()), columns=["columna", "nulos"])
reporte_nulos["porcentaje_nulos"] = (reporte_nulos["nulos"] / total_registros) * 100

display(reporte_nulos.sort_values("porcentaje_nulos", ascending=False))

Se evidencia la presencia de valores faltantes en múltiples variables del conjunto de datos, con comportamientos diferenciados según el tipo de información. En particular, las variables asociadas a los resultados académicos (PUNT_GLOBAL, PUNT_LECTURA_CRITICA, PUNT_C_NATURALES y PUNT_SOCIALES_CIUDADANAS) presentan un porcentaje alto de valores nulos (≈38.6%), lo cual sugiere que estos datos no estaban disponibles para ciertos periodos del dataset, más que tratarse de datos faltantes aleatorios.

Por otro lado, las variables de contexto socioeconómico, como FAMI_TIENEINTERNET, FAMI_TIENECOMPUTADOR y FAMI_ESTRATOVIVIENDA, presentan porcentajes de valores nulos más bajos (entre 2% y 3%), lo que indica una falta de completitud parcial en la información reportada por los estudiantes.

En general, se observa que la calidad de los datos no es homogénea entre variables, existiendo tanto valores faltantes estructurales (asociados a cambios en la recolección de datos a lo largo del tiempo) como valores faltantes puntuales (COLE_BILINGUE, Educación de los padres), lo cual deberá ser considerado en las etapas posteriores de limpieza y análisis.

### 3.2 Técnicas propuestas para tratar valores faltantes

Para el tratamiento de valores faltantes se proponen diferentes estrategias según el tipo de variable, el porcentaje de datos faltantes y su relevancia dentro del análisis.

- En primer lugar, las variables académicas (PUNT_GLOBAL y los puntajes por área) presentan un alto porcentaje de valores nulos (≈38.6%), lo cual no corresponde a datos faltantes aleatorios, sino a periodos en los que esta información no era registrada. Por esta razón, no se realizará imputación sobre estos valores, sino que se trabajará únicamente con los periodos en los que los puntajes estén disponibles, garantizando consistencia en los análisis.
- En el caso de la variable COLE_BILINGUE, que presenta un porcentaje de valores nulos relativamente alto (≈8.1%), se evaluará mantener los valores faltantes como una categoría independiente (por ejemplo, “SIN INFORMACIÓN”), dado que la ausencia de este dato puede estar asociada a diferencias en el reporte institucional y no necesariamente a una condición inexistente.
- Para las variables de contexto socioeconómico, como FAMI_EDUCACIONPADRE y FAMI_EDUCACIONMADRE (≈3.5% de nulos), así como FAMI_ESTRATOVIVIENDA, FAMI_TIENECOMPUTADOR, FAMI_TIENEINTERNET, FAMI_PERSONASHOGAR, FAMI_TIENEAUTOMOVIL, FAMI_CUARTOSHOGAR y FAMI_TIENELAVADORA (≈1%–3%), se considerará la imputación mediante una categoría adicional (“SIN INFORMACIÓN”), permitiendo conservar los registros y evitando introducir supuestos fuertes que puedan sesgar el análisis.
- Por otro lado, las variables con un porcentaje muy bajo de valores nulos (menor al 1%), podrán ser tratadas mediante eliminación de registros incompletos o imputación simple, dado que su impacto en el análisis será mínimo.

Finalmente, se considerará la eliminación de variables que no aporten valor analítico o que correspondan a identificadores (por ejemplo, códigos únicos), así como la aplicación de transformaciones adicionales en etapas posteriores para mejorar la calidad y consistencia del dataset.

### 3.3 Detección de inconsistencias

In [0]:
# Valores distintos (para detectar inconsistencias)
display(dfBoyaca.select("COLE_MCPIO_UBICACION").distinct())

El análisis de valores únicos en variables geográficas como COLE_MCPIO_UBICACION y ESTU_MCPIO_RESIDE evidenció la presencia de inconsistencias en la representación de los nombres de municipios, principalmente asociadas a diferencias en el uso de tildes y formatos de escritura (por ejemplo, “SANTA MARIA” y “SANTA MARÍA”), estas inconsistencias generan duplicados afectando los resultados de los análisis agregados.

Dado que el número de municipios es elevado, la corrección manual de estos valores no es viable. Por lo tanto, se propone realizar un proceso de estandarización automática de texto, el cual incluya la normalización de caracteres (eliminación o unificación de tildes) y conversión a mayúsculas, para consolidar las categorías equivalentes en una única representación consistente.

## 4. Planteamiento de preguntas de negocio

Con base en el análisis exploratorio y el objetivo de mejorar los resultados de las pruebas Saber 11 en el departamento de Boyacá, se plantean preguntas orientadas a identificar factores clave de desempeño y posibles líneas de acción.

**Factores socioeconómicos**

1. ¿Qué variables socioeconómicas (estrato, acceso a internet, disponibilidad de computador, educación de los padres) están más asociadas con mayores puntajes en Saber 11 en Boyacá?
2. ¿Cómo cambia el desempeño académico cuando se combinan condiciones del hogar (por ejemplo, internet + computador + nivel educativo de los padres)?

**Brechas y priorización territorial**

3. ¿Qué municipios de Boyacá presentan mayores brechas de desempeño y cuáles concentran los mejores resultados?
4. ¿Existen diferencias consistentes entre zonas urbanas y rurales, y en qué magnitud se presentan?
5. ¿Qué características institucionales (naturaleza oficial/no oficial, jornada, bilingüismo, zona urbana/rural) se asocian con mejores resultados en Boyacá?

**Movilidad y acceso educativo**

6. ¿Existen diferencias en el desempeño entre estudiantes que residen en Boyacá y estudian en el mismo departamento versus aquellos que estudian fuera?
7. ¿Qué perfil socioeconómico y académico tienen los estudiantes que salen de Boyacá a estudiar en otras regiones, y cómo se comparan con quienes permanecen?

**Tendencias y estabilidad**

8. ¿Cómo ha evolucionado el desempeño académico en Boyacá en los periodos con datos disponibles y qué tendencias se observan en los grupos de mayor y menor rendimiento?

Las preguntas buscan no solo describir el comportamiento de los datos, sino identificar factores accionables que orienten decisiones de política pública y focalización de recursos.

## 5. Filtros, limpieza y transformación inicial

Se realizan algunos filtros, procesos de limpieza y transformaciones iniciales sobre el conjunto de datos, con el objetivo de mejorar su calidad y preparar la información para análisis posteriores.

### 5.1 Filtrado del dataset

Se filtraron los registros asociados al departamento de Boyacá considerando tanto la ubicación del establecimiento educativo como la residencia del estudiante, con el fin de capturar de manera amplia la población relevante para el análisis.

In [0]:
# Filtrar datos relacionados con Boyacá
dfClean = dfPruebasSaber.filter(
    (col("COLE_DEPTO_UBICACION") == "BOYACA") |
    (col("ESTU_DEPTO_RESIDE") == "BOYACA")
)
print("Registros después del filtro:", dfClean.count())

### 5.2 Tratamiento de valores faltantes

In [0]:
#1. Variables académicas (NO imputar)
# Filtrar registros con puntaje global válido
dfClean = dfClean.filter(col("PUNT_GLOBAL").isNotNull())

Se identificó que la mayoría de variables de puntaje académico presentan valores nulos de manera consistente entre sí, lo que indica que corresponden a registros en los cuales no se dispone de resultados de la prueba. Por esta razón, se utilizó la variable `PUNT_GLOBAL` como referencia para filtrar registros válidos, dado que su ausencia implica la falta de información en la mayoría de variables de puntaje.

In [0]:
#2. Variables categóricas → “SIN INFORMACIÓN”
dfClean = dfClean.fillna({
    "FAMI_TIENEINTERNET": "SIN INFORMACION",
    "FAMI_TIENECOMPUTADOR": "SIN INFORMACION",
    "FAMI_ESTRATOVIVIENDA": "SIN INFORMACION",
    "FAMI_EDUCACIONPADRE": "SIN INFORMACION",
    "FAMI_EDUCACIONMADRE": "SIN INFORMACION",
    "COLE_BILINGUE": "SIN INFORMACION",
    "FAMI_PERSONASHOGAR": "SIN INFORMACION",
    "FAMI_CUARTOSHOGAR": "SIN INFORMACION",
    "FAMI_TIENEAUTOMOVIL": "SIN INFORMACION",
    "FAMI_TIENELAVADORA": "SIN INFORMACION"
})

In [0]:
#3. Variables con pocos nulos → eliminación
dfClean = dfClean.dropna(subset=["ESTU_GENERO", "COLE_AREA_UBICACION", "COLE_CODIGO_ICFES", "ESTU_COD_RESIDE_DEPTO", "ESTU_MCPIO_RESIDE", "ESTU_COD_RESIDE_MCPIO", "ESTU_DEPTO_RESIDE", "COLE_CARACTER", "COLE_CALENDARIO", "COLE_COD_DANE_ESTABLECIMIENTO", "DESEMP_INGLES", "COLE_SEDE_PRINCIPAL", "COLE_JORNADA", "COLE_COD_DANE_SEDE", "COLE_COD_DEPTO_UBICACION", "COLE_COD_MCPIO_UBICACION", "COLE_DEPTO_UBICACION", "COLE_GENERO", "COLE_NOMBRE_ESTABLECIMIENTO", "COLE_NOMBRE_SEDE", "COLE_NATURALEZA", "COLE_MCPIO_UBICACION", "PUNT_INGLES"])

Se eliminaron registros con valores faltantes en variables con muy baja proporción de nulos, dado que su impacto sobre el tamaño del dataset es mínimo y permite mantener la consistencia de la información.

### 5.3 Estandarización de variables categóricas

In [0]:
import unicodedata
# Función para quitar tildes
def quitar_tildes(texto):
    if texto is None:
        return None
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
quitar_tildes_udf = F.udf(quitar_tildes)

In [0]:
#Aplicamos a municipios:
dfClean = dfClean.withColumn(
    "COLE_MCPIO_UBICACION",
    upper(trim(quitar_tildes_udf(col("COLE_MCPIO_UBICACION"))))
)

dfClean = dfClean.withColumn(
    "ESTU_MCPIO_RESIDE",
    upper(trim(quitar_tildes_udf(col("ESTU_MCPIO_RESIDE"))))
)

Se realizó un proceso de estandarización de variables geográficas mediante la normalización de texto, eliminación de tildes y conversión a mayúsculas, con el fin de evitar duplicidades en la representación de municipios.

### 5.4 Transformaciones iniciales (valor analítico)

In [0]:
# 1. Clasificación del puntaje
dfClean = dfClean.withColumn(
    "NIVEL_DESEMPENO",
    when(col("PUNT_GLOBAL") < 200, "BAJO")
    .when((col("PUNT_GLOBAL") >= 200) & (col("PUNT_GLOBAL") < 300), "MEDIO")
    .otherwise("ALTO")
)

In [0]:
# 2. Indicador de acceso digital
dfClean = dfClean.withColumn(
    "ACCESO_TECNOLOGICO",
    when(
        (col("FAMI_TIENEINTERNET") == "Si") &
        (col("FAMI_TIENECOMPUTADOR") == "Si"),
        "ALTO"
    ).otherwise("LIMITADO")
)

Se crearon nuevas variables que permiten facilitar el análisis del desempeño académico. En particular, se generó una clasificación del puntaje global y un indicador de acceso tecnológico, lo cual permitirá evaluar de manera más directa la relación entre condiciones del hogar y resultados académicos en etapas posteriores.

### 5.5 Vista final

In [0]:
print("Total de registros: ", dfClean.count())
display(dfClean.limit(10))

In [0]:
#Verificar nulos
import pandas as pd
total_reg = dfClean.count()

dfNull = dfClean.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in dfClean.columns
])
nul_dict = dfNull.collect()[0].asDict()

reporte_nul = pd.DataFrame(list(nul_dict.items()), columns=["columna", "nulos"])
reporte_nul["porcentaje_nulos"] = (reporte_nul["nulos"] / total_reg) * 100

display(reporte_nul.sort_values("porcentaje_nulos", ascending=False))

No hay presencia de valores faltantes en las columnas del dataset.

In [0]:
#Verficar estandarización
display(dfClean.select("COLE_MCPIO_UBICACION").distinct())

In [0]:
df_limpio = dfClean.filter(col("COLE_COD_DEPTO_UBICACION") == 15)
display(df_limpio.select("COLE_MCPIO_UBICACION").distinct())

In [0]:
display(df_limpio)

Después de la estandarización para evitar duplicados en los municipios, la variable `COLE_MCPIO_UBICACION` redujo sus clases de 243 a 124 municipios.

### Promedio de puntaje global por municipio del colegio

In [0]:
dfMunicipioPuntaje = (
    df_limpio
    .groupBy("COLE_MCPIO_UBICACION")
    .agg(
        F.avg("PUNT_GLOBAL").alias("promedio_puntaje_global"),
        F.count("PUNT_GLOBAL").alias("cantidad_con_puntaje"),
        F.count("*").alias("total_registros")
    )
    .orderBy(F.col("promedio_puntaje_global").desc())
)
display(dfMunicipioPuntaje.limit(10))

En un primer análisis del puntaje global promedio por municipio, se identificó que algunos de los municipios con mayor promedio no pertenecen al departamento de Boyacá. Este comportamiento se debe a que el conjunto de datos incluye estudiantes que residen en Boyacá, pero que presentan la prueba en instituciones ubicadas en otros departamentos.
Adicionalmente, se observa que estos municipios presentan una cantidad muy baja de registros, lo que genera promedios poco representativos y puede distorsionar la interpretación de los resultados.

**Municipios con mejor desempeño**

In [0]:
dfTopMunicipios = (
    dfMunicipioPuntaje
    .filter(col("cantidad_con_puntaje") >= 30)   # para evitar municipios con muy pocos casos
    .orderBy(col("promedio_puntaje_global").desc())
)
display(dfTopMunicipios)

Con el fin de obtener resultados más representativos, se aplicó un filtro considerando únicamente municipios con al menos 30 registros con puntaje disponible. A partir de este ajuste, los municipios con mejor desempeño corresponden en su totalidad al departamento de Boyacá, permitiendo un análisis más consistente del comportamiento académico a nivel territorial.

**Municipios con peor desempeño**

In [0]:
dfBottomMunicipios = (
    dfMunicipioPuntaje
    .filter(col("cantidad_con_puntaje") >= 30)   # para evitar municipios con muy pocos casos
    .orderBy(col("promedio_puntaje_global").asc())
)
display(dfBottomMunicipios)